# NB03b — Qwen Menu Understanding on the Curated Subset

This notebook is the **first true Phase 3 understanding pass**. It uses the fixed subset
created in `NB03a` and asks a tightly scoped question:

> can a small mixed-quality public menu-image subset still yield useful structured
> understanding evidence on local hardware?

The purpose is not to build a production extractor. It is to produce enough direct
evidence for the horizon scan to judge whether a multimodal understanding layer is
practical and worthwhile before any future image-generation tuning.

Main outputs:

- `data/processed/menu_image_subset_v1_qwen_understanding.csv`
- `data/processed/menu_image_subset_v1_qwen_understanding.jsonl`

Run cells top-to-bottom. If the Qwen multimodal projector file is missing, stop after the
readiness check and fetch that sidecar before continuing.

## 1. Environment setup

This notebook stays intentionally small:

- one fixed subset only: `menu_image_subset_v1.csv`
- one local VLM only: `qwen25_vl_7b`
- one structured extraction pass only

The result is a compact evidence table, not a benchmark suite.

In [ ]:
from __future__ import annotations

import base64
import csv
import gc
import json
import math
import mimetypes
import time
from io import BytesIO
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import yaml
from PIL import Image, ImageOps

ROOT = Path().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "configs").exists(), f"Cannot find project root from {Path().resolve()}"

PROCESSED_DIR = ROOT / "data" / "processed"
SUBSET_PATH = PROCESSED_DIR / "menu_image_subset_v1.csv"
OUTPUT_CSV = PROCESSED_DIR / "menu_image_subset_v1_qwen_understanding.csv"
OUTPUT_JSONL = PROCESSED_DIR / "menu_image_subset_v1_qwen_understanding.jsonl"
MODEL_REGISTRY = ROOT / "configs" / "model_registry.yaml"
FIG_DIR = ROOT / "outputs" / "figures" / "nb03b"
FIG_DIR.mkdir(parents=True, exist_ok=True)

assert SUBSET_PATH.exists(), f"Subset file not found: {SUBSET_PATH}"

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({"figure.dpi": 150, "savefig.bbox": "tight"})

print(f"Project root : {ROOT}")
print(f"Subset path  : {SUBSET_PATH}")
print(f"Output CSV   : {OUTPUT_CSV}")
print(f"Output JSONL : {OUTPUT_JSONL}")

## 2. Load the fixed Phase 3 subset

The subset is already accepted as **good enough** for this sandbox. We are not revisiting
manual `keep_core / keep_edge / drop` work here. The analytical question now is whether
Qwen can turn this small mixed-quality set into denser multimodal evidence.

In [ ]:
subset = pd.read_csv(SUBSET_PATH).copy()

print(f"Subset size: {len(subset)}")
print()
print("By source:")
print(subset["source"].value_counts().to_string())
print()
print("By subset role:")
print(subset["subset_role"].value_counts().to_string())
print()

display(
    subset[["subset_role", "source", "filename", "final_score", "selection_mode"]]
    .sort_values(["subset_role", "source", "final_score"], ascending=[True, True, False])
    .reset_index(drop=True)
)

## 3. Qwen readiness check

For `llama-cpp-python`, Qwen image inference needs **two** local assets:

1. the main Qwen GGUF
2. the separate multimodal projector (`mmproj`) sidecar

If the second file is missing, the notebook should fail **early and clearly** rather than
halfway through the extraction loop.

In [ ]:
with open(MODEL_REGISTRY, "r", encoding="utf-8") as f:
    registry = yaml.safe_load(f)["models"]

qwen_cfg = registry["qwen25_vl_7b"]

model_path_raw = Path(qwen_cfg["path"])
mmproj_path_raw = Path(qwen_cfg.get("mmproj_path", "models/mmproj-Qwen_Qwen2.5-VL-7B-Instruct-f16.gguf"))

QWEN_MODEL_PATH = model_path_raw if model_path_raw.is_absolute() else ROOT / model_path_raw
QWEN_MMPROJ_PATH = mmproj_path_raw if mmproj_path_raw.is_absolute() else ROOT / mmproj_path_raw

status_df = pd.DataFrame([
    {
        "asset": "qwen25_vl_7b gguf",
        "path": str(QWEN_MODEL_PATH),
        "exists": QWEN_MODEL_PATH.exists(),
    },
    {
        "asset": "qwen25_vl_7b mmproj",
        "path": str(QWEN_MMPROJ_PATH),
        "exists": QWEN_MMPROJ_PATH.exists(),
    },
])

display(status_df)

if not QWEN_MMPROJ_PATH.exists():
    print("Missing multimodal sidecar:")
    print(QWEN_MMPROJ_PATH)
    print()
    print("This is expected if only the main GGUF was downloaded earlier.")
    print("Do not continue into image inference until the mmproj file exists locally.")

READY_FOR_VISION = bool(QWEN_MODEL_PATH.exists() and QWEN_MMPROJ_PATH.exists())
print()
print({"ready_for_qwen_vision": READY_FOR_VISION})

Interpretation:

- `READY_FOR_VISION = True` means the notebook can proceed with image understanding
- `READY_FOR_VISION = False` means local Qwen setup is still incomplete for multimodal use

That is a setup issue, not a research failure. It simply means the current local model
bundle is missing one required inference asset.

## 4. Prompt and output schema

The schema stays deliberately compact. Each image gets:

- `ocr_like_text`
- `dense_caption`
- `layout_note`
- `style_tags`
- `page_type`
- `extraction_utility`
- `quality_confidence_note`

The wording is intentionally conservative. If text is unreadable, the model should say so
instead of inventing menu content.

In [ ]:
SYSTEM_PROMPT = """You are assisting a research sandbox that evaluates menu-image understanding.
Be cautious, literal, and non-hallucinatory.
Only report text or structure that is plausibly visible in the image.
If the text is unclear, say so plainly rather than guessing.
Do not be overconfident: partial readability should not be described as fully clear or fully accurate."""

USER_PROMPT = """Return one JSON object with exactly these keys:
- ocr_like_text
- dense_caption
- layout_note
- style_tags
- page_type
- extraction_utility
- quality_confidence_note

Rules:
- `ocr_like_text`: short visible text fragments only; use [unclear] for partial text.
- keep `ocr_like_text` concise: no more than 6 short fragments or about 50 words total.
- `dense_caption`: 1-2 sentences describing the page content.
- `layout_note`: brief note on columns, sections, prices, headings, handwriting, decorations, or crop issues.
- `style_tags`: array of 3-6 short snake_case tags.
- `page_type`: one of `menu_page`, `menu_adjacent`, `ambiguous`.
- `extraction_utility`: one of `high`, `medium`, `low`.
- choose `high` only when this is clearly a true menu page with multiple readable menu elements.
- choose `medium` when it is partly readable, structurally useful, or menu-adjacent.
- choose `low` when text is sparse, ambiguous, decorative, badly cropped, or hard to trust.
- if `page_type` is `ambiguous`, do not use `high`.
- `quality_confidence_note`: explain legibility and confidence in one short note using cautious wording.

Do not invent dish names, prices, or restaurant details that are not visibly supported.
Return JSON only."""

RETRY_USER_PROMPT = """Return one compact JSON object with exactly these keys:
- ocr_like_text
- dense_caption
- layout_note
- style_tags
- page_type
- extraction_utility
- quality_confidence_note

Hard limits:
- `ocr_like_text`: maximum 4 short fragments, about 30 words total
- `dense_caption`: 1 sentence
- `layout_note`: 1 short sentence
- `style_tags`: 3 to 5 short tags
- `quality_confidence_note`: 1 short sentence

Do not include any explanation outside JSON.
Return valid JSON only."""


EXPECTED_KEYS = [
    "ocr_like_text",
    "dense_caption",
    "layout_note",
    "style_tags",
    "page_type",
    "extraction_utility",
    "quality_confidence_note",
]

## 5. Helper functions

Two implementation choices matter here:

1. local images are converted to `data:` URIs before sending to Qwen
2. very large images are resized on the fly, with a stricter cap for Wikimedia, so 8k pages stay realistic for this hardware
3. results are saved incrementally so an interrupted run does not lose the whole notebook pass

In [ ]:
DEFAULT_MAX_IMAGE_EDGE = 1920
WIKIMEDIA_MAX_IMAGE_EDGE = 1280
JPEG_QUALITY = 90


def choose_max_edge(source: str) -> int:
    return WIKIMEDIA_MAX_IMAGE_EDGE if source == "wikimedia" else DEFAULT_MAX_IMAGE_EDGE


def image_to_data_uri(
    path: str | Path,
    source: str,
    max_edge: int | None = None,
) -> tuple[str, tuple[int, int], tuple[int, int], int]:
    path = Path(path)
    if max_edge is None:
        max_edge = choose_max_edge(source)
    with Image.open(path) as im:
        im = ImageOps.exif_transpose(im).convert("RGB")
        original_size = im.size
        if max(im.size) > max_edge:
            im.thumbnail((max_edge, max_edge))
        resized_size = im.size
        buf = BytesIO()
        im.save(buf, format="JPEG", quality=JPEG_QUALITY)
    encoded = base64.b64encode(buf.getvalue()).decode("utf-8")
    return f"data:image/jpeg;base64,{encoded}", original_size, resized_size, max_edge


def parse_json_object(text: str) -> dict:
    text = (text or "").strip()
    start = text.find("{")
    end = text.rfind("}")
    if start == -1 or end == -1 or end <= start:
        raise ValueError(f"No JSON object found in response: {text[:200]}")
    candidate = text[start:end + 1]
    try:
        return json.loads(candidate)
    except json.JSONDecodeError:
        candidate = candidate.replace("\r", " ").replace("\n", " ")
        return json.loads(candidate)


def normalize_payload(payload: dict) -> dict:
    payload = dict(payload)
    for key in EXPECTED_KEYS:
        payload.setdefault(key, None)

    ocr_like_text = payload.get("ocr_like_text")
    if isinstance(ocr_like_text, list):
        ocr_like_text = " ".join(str(part).strip() for part in ocr_like_text if str(part).strip())
    elif ocr_like_text is not None:
        ocr_like_text = str(ocr_like_text).strip()
    payload["ocr_like_text"] = ocr_like_text or None

    for key in ["dense_caption", "layout_note", "quality_confidence_note"]:
        value = payload.get(key)
        payload[key] = str(value).strip() if value is not None and str(value).strip() else None

    style_tags = payload.get("style_tags")
    if isinstance(style_tags, str):
        style_tags = [part.strip() for part in style_tags.split(",") if part.strip()]
    elif not isinstance(style_tags, list):
        style_tags = []

    payload["style_tags"] = [str(tag).strip() for tag in style_tags if str(tag).strip()]
    payload["page_type"] = str(payload.get("page_type") or "").strip().lower() or None
    payload["extraction_utility"] = str(payload.get("extraction_utility") or "").strip().lower() or None

    return {key: payload.get(key) for key in EXPECTED_KEYS}


def extract_or_retry(
    llm,
    image_uri: str,
    prompt_text: str,
    max_tokens: int,
    seed: int,
) -> tuple[dict, dict]:
    response = llm.create_chat_completion(
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {
                "role": "user",
                "content": [
                    {"type": "image_url", "image_url": {"url": image_uri}},
                    {"type": "text", "text": prompt_text},
                ],
            },
        ],
        temperature=0.0,
        top_p=0.9,
        max_tokens=max_tokens,
        seed=seed,
        response_format={"type": "json_object"},
    )
    payload = normalize_payload(parse_json_object(response["choices"][0]["message"]["content"]))
    return payload, response


def write_jsonl(records: list[dict], path: Path) -> None:
    with open(path, "w", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")


def records_to_frame(records: list[dict]) -> pd.DataFrame:
    rows = []
    for record in records:
        row = dict(record)
        style_tags = row.get("style_tags")
        if isinstance(style_tags, list):
            row["style_tags"] = " | ".join(style_tags)
        rows.append(row)
    return pd.DataFrame(rows)


def load_existing_records(path: Path) -> list[dict]:
    if not path.exists():
        return []

    existing = pd.read_csv(path)
    records = []
    for row in existing.to_dict(orient="records"):
        style_tags = row.get("style_tags")
        if isinstance(style_tags, str) and style_tags.strip():
            row["style_tags"] = [part.strip() for part in style_tags.split("|") if part.strip()]
        else:
            row["style_tags"] = []
        records.append(row)
    return records

## 6. Load Qwen

This notebook assumes the **single-model-at-a-time** policy established for the project.
Load Qwen, run the subset sequentially, save outputs, then unload the model.

In [ ]:
assert READY_FOR_VISION, (
    "Qwen vision assets are incomplete. "
    f"Expected gguf at {QWEN_MODEL_PATH} and mmproj at {QWEN_MMPROJ_PATH}."
)

from llama_cpp import Llama
from llama_cpp.llama_chat_format import Qwen25VLChatHandler

chat_handler = Qwen25VLChatHandler(
    clip_model_path=str(QWEN_MMPROJ_PATH),
    verbose=False,
)

llm = Llama(
    model_path=str(QWEN_MODEL_PATH),
    chat_handler=chat_handler,
    n_ctx=max(int(qwen_cfg.get("context_length", 4096)), 4096),
    n_gpu_layers=int(qwen_cfg.get("n_gpu_layers", -1)),
    n_threads=int(qwen_cfg.get("n_threads", 8)),
    verbose=False,
)

print({
    "model_path": str(QWEN_MODEL_PATH),
    "mmproj_path": str(QWEN_MMPROJ_PATH),
    "n_ctx": max(int(qwen_cfg.get("context_length", 4096)), 4096),
})

## 7. Run the structured understanding pass

This loop is intentionally small and auditable:

- one image at a time
- deterministic settings
- incremental save after each record
- explicit error capture for weak or problematic pages

In [ ]:
def run_qwen_on_image(llm, image_path: str | Path, source: str) -> tuple[dict, dict]:
    image_uri, original_size, resized_size, max_edge = image_to_data_uri(image_path, source=source)

    try:
        payload, response = extract_or_retry(
            llm=llm,
            image_uri=image_uri,
            prompt_text=USER_PROMPT,
            max_tokens=320,
            seed=42,
        )
    except Exception:
        payload, response = extract_or_retry(
            llm=llm,
            image_uri=image_uri,
            prompt_text=RETRY_USER_PROMPT,
            max_tokens=220,
            seed=43,
        )

    payload["original_w"] = original_size[0]
    payload["original_h"] = original_size[1]
    payload["inference_w"] = resized_size[0]
    payload["inference_h"] = resized_size[1]
    payload["inference_max_edge"] = max_edge
    return payload, response


OVERWRITE_EXISTING_RESULTS = True

existing_records = [] if OVERWRITE_EXISTING_RESULTS else load_existing_records(OUTPUT_CSV)
records_by_path = {record["path"]: record for record in existing_records if "path" in record}

pending = subset.loc[~subset["path"].isin(records_by_path)].copy()

print(f"Overwrite existing results: {OVERWRITE_EXISTING_RESULTS}")
print(f"Existing records : {len(records_by_path)}")
print(f"Pending images   : {len(pending)}")

for idx, (_, row) in enumerate(pending.iterrows(), start=1):
    print(f"[{idx}/{len(pending)}] {row['source']} | {row['filename']}")
    t0 = time.perf_counter()

    base_record = {
        "path": row["path"],
        "filename": row["filename"],
        "source": row["source"],
        "subset_role": row.get("subset_role"),
        "selection_mode": row.get("selection_mode"),
        "final_score": row.get("final_score"),
        "model_name": "qwen25_vl_7b",
    }

    try:
        payload, response = run_qwen_on_image(llm, row["path"], row["source"])
        usage = response.get("usage", {})
        base_record.update(payload)
        base_record.update({
            "run_status": "ok",
            "elapsed_s": round(time.perf_counter() - t0, 2),
            "prompt_tokens": usage.get("prompt_tokens"),
            "completion_tokens": usage.get("completion_tokens"),
            "total_tokens": usage.get("total_tokens"),
            "finish_reason": response["choices"][0].get("finish_reason"),
            "raw_response_json": response["choices"][0]["message"]["content"],
            "error": None,
        })
    except Exception as exc:
        base_record.update({
            "ocr_like_text": None,
            "dense_caption": None,
            "layout_note": None,
            "style_tags": [],
            "page_type": None,
            "extraction_utility": None,
            "quality_confidence_note": None,
            "run_status": "error",
            "elapsed_s": round(time.perf_counter() - t0, 2),
            "prompt_tokens": None,
            "completion_tokens": None,
            "total_tokens": None,
            "finish_reason": None,
            "raw_response_json": None,
            "error": str(exc),
        })

    records_by_path[row["path"]] = base_record

    ordered_records = [
        records_by_path[path]
        for path in subset["path"]
        if path in records_by_path
    ]

    write_jsonl(ordered_records, OUTPUT_JSONL)
    records_to_frame(ordered_records).to_csv(
        OUTPUT_CSV,
        index=False,
        quoting=csv.QUOTE_MINIMAL,
    )

print()
print(f"Saved CSV   : {OUTPUT_CSV}")
print(f"Saved JSONL : {OUTPUT_JSONL}")

## 8. Inspect the saved output

This is the direct evidence table for the horizon scan. A useful outcome here is **not**
perfect OCR. It is enough if:

- stronger pages yield recognisable text and structure
- weaker Yelp images mostly yield low-confidence or low-utility notes
- the output is rich enough to support later JEPA-side embedding analysis

In [ ]:
results = pd.read_csv(OUTPUT_CSV)

size_defaults = (
    subset[["path", "source", "download_w", "download_h"]]
    .rename(columns={"download_w": "original_w", "download_h": "original_h"})
    .copy()
)
size_defaults["inference_w"] = size_defaults["original_w"]
size_defaults["inference_h"] = size_defaults["original_h"]
size_defaults["inference_max_edge"] = size_defaults["source"].map(choose_max_edge)
size_defaults = size_defaults.drop(columns=["source"])

missing_size_cols = [
    col for col in ["original_w", "original_h", "inference_w", "inference_h", "inference_max_edge"]
    if col not in results.columns
]

if missing_size_cols:
    results = results.merge(size_defaults, on="path", how="left")
    print("Backfilled legacy size columns from subset metadata:")
    print(", ".join(missing_size_cols))
    print("These rows were produced before the resized-inference fields were added.")
    print()

display(
    results[[
        "source",
        "subset_role",
        "filename",
        "original_w",
        "original_h",
        "inference_w",
        "inference_h",
        "inference_max_edge",
        "page_type",
        "extraction_utility",
        "quality_confidence_note",
        "ocr_like_text",
        "dense_caption",
        "layout_note",
        "style_tags",
        "run_status",
        "elapsed_s",
    ]]
)

print("Run status:")
print(results["run_status"].value_counts(dropna=False).to_string())
print()

if "extraction_utility" in results.columns:
    print("Extraction utility by source:")
    print(pd.crosstab(results["source"], results["extraction_utility"], dropna=False).to_string())
    print()

if "page_type" in results.columns:
    print("Page type by source:")
    print(pd.crosstab(results["source"], results["page_type"], dropna=False).to_string())

## 9. Optional unload

Unload the model before moving into any later notebook so the project stays aligned with
the single-model-at-a-time hardware policy.

In [ ]:
try:
    del llm
except NameError:
    pass

gc.collect()
print("Qwen handle released.")

## 10. Visualise the evidence

These figures are intentionally small and interpretive. They are not benchmark graphics.
Their role is to make the **pattern of evidence** visible:

- which sources completed cleanly
- which sources produced higher-utility understanding outputs
- what the runtime looked like on local hardware
- which concrete images count as useful or fragile evidence cases

In [ ]:
viz = results.copy()

viz["extraction_utility_plot"] = (
    viz["extraction_utility"]
    .fillna("missing")
    .astype(str)
    .str.strip()
    .str.lower()
    .replace({"moderate": "medium"})
)

viz["page_type_plot"] = (
    viz["page_type"]
    .fillna("missing")
    .astype(str)
    .str.strip()
    .str.lower()
    .replace({"restaurant menu": "menu_page"})
)

viz["run_status_plot"] = viz["run_status"].fillna("missing").astype(str).str.strip().str.lower()

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

run_order = ["ok", "error", "missing"]
util_order = ["high", "medium", "low", "missing"]
page_order = ["menu_page", "ambiguous", "menu_adjacent", "missing"]

run_ct = pd.crosstab(viz["source"], viz["run_status_plot"]).reindex(columns=run_order, fill_value=0)
util_ct = pd.crosstab(viz["source"], viz["extraction_utility_plot"]).reindex(columns=util_order, fill_value=0)
time_df = viz.loc[viz["elapsed_s"].notna(), ["source", "elapsed_s"]].copy()

run_ct.plot(kind="bar", stacked=True, ax=axes[0], color=["#4c956c", "#c1121f", "#adb5bd"])
axes[0].set_title("Run status by source")
axes[0].set_xlabel("")
axes[0].set_ylabel("images")
axes[0].legend(title="", fontsize=8)

util_ct.plot(kind="bar", stacked=True, ax=axes[1], color=["#2a9d8f", "#e9c46a", "#f4a261", "#adb5bd"])
axes[1].set_title("Extraction utility by source")
axes[1].set_xlabel("")
axes[1].set_ylabel("images")
axes[1].legend(title="", fontsize=8)

sns.boxplot(data=time_df, x="source", y="elapsed_s", ax=axes[2], color="#90caf9", showfliers=False)
sns.stripplot(data=time_df, x="source", y="elapsed_s", ax=axes[2], color="#1d3557", size=5, alpha=0.8)
axes[2].set_title("Elapsed seconds by source")
axes[2].set_xlabel("")
axes[2].set_ylabel("seconds")

for ax in axes:
    ax.tick_params(axis="x", rotation=15)

fig.suptitle("Qwen understanding evidence on the curated Phase 3 subset", fontsize=14)
fig.tight_layout()
summary_path = FIG_DIR / "fig23_qwen_understanding_summary.png"
fig.savefig(summary_path)
plt.show()

print(f"Saved figure: {summary_path}")

In [ ]:
contact = viz.copy()

cols = 4
rows = math.ceil(len(contact) / cols)
fig, axes = plt.subplots(rows, cols, figsize=(16, 4.8 * rows))
axes = np.array(axes).reshape(rows, cols)

for ax in axes.ravel():
    ax.axis("off")

for ax, (_, row) in zip(axes.ravel(), contact.iterrows()):
    with Image.open(row["path"]) as im:
        im = ImageOps.exif_transpose(im).convert("RGB")
        thumb = im.copy()
        thumb.thumbnail((900, 520))
    ax.imshow(thumb)
    ax.axis("off")

    utility = row.get("extraction_utility_plot", "missing")
    status = row.get("run_status_plot", "missing")
    page_type = row.get("page_type_plot", "missing")
    note = str(row.get("quality_confidence_note") or "no confidence note")
    note = note[:95] + "..." if len(note) > 95 else note

    ax.set_title(
        "\n".join([
            f"{row['source']} | {row['filename']}",
            f"status={status} | utility={utility} | page={page_type}",
            note,
        ]),
        fontsize=8,
    )

fig.suptitle("Annotated contact sheet: current Qwen evidence cases", fontsize=14)
fig.tight_layout()
contact_path = FIG_DIR / "fig24_qwen_understanding_contact_sheet.png"
fig.savefig(contact_path)
plt.show()

print(f"Saved figure: {contact_path}")

## 11. Phase 3 handoff note

`NB03b` is complete when the saved understanding table shows whether the curated subset is
semantically useful or mostly too weak.

The next notebook should use this output as a **supporting semantic layer**, not as ground
truth:

- `NB03c`: frozen JEPA embeddings on the same subset
- compare whether Qwen-rich pages cluster differently from weak edge cases
- use the joint result to support the argument that understanding can densify supervision
  before any future image-generation tuning